# Prerequisites
- Python 3.10.7 or greater
- A Google Cloud project
- A Google account with Gmail enabled.

# Steps

1. Create Google Cloud Project
2. Enable Email API
3. Configure the OAuth consent screen: Menu  -> Google Auth platform -> Branding    
4. Authorize credentials for a desktop application
5. Create Client: Menu > Google Auth platform > Clients
6. Save the downloaded JSON file as client_secret.json, and move the file to your working directory.
7. Install the Google client library: 
 ```
 pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

 ```


create service and authenticate

In [ ]:
import os
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from pathlib import Path


def create_service(client_secret_file, api_name, api_version, *scopes, prefix=''):
    
    CLIENT_SECRET_FILE = client_secret_file
    API_SERVICE_NAME = api_name
    API_VERSION = api_version
    SCOPES = [scope for scope in scopes][0]

    creds = None
    working_dir = os.getcwd()
    token_dir = 'token files'
    token_file = f'token_{API_SERVICE_NAME}_{API_VERSION}{prefix}.json'

    ### Check if token dir exists first, if not, create the folder
    if not os.path.exists(os.path.join(working_dir, token_dir)):
        os.mkdir(os.path.join(working_dir, token_dir))

    if os.path.exists(os.path.join(working_dir, token_dir, token_file)):
        creds = Credentials.from_authorized_user_file(
            os.path.join(working_dir, token_dir, token_file), SCOPES
        )

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CLIENT_SECRET_FILE, SCOPES)
            creds = flow.run_local_server(port=0)

        with open(os.path.join(working_dir, token_dir, token_file), 'w') as token:
            token.write(creds.to_json())

    try:
        service = build(
            API_SERVICE_NAME,
            API_VERSION,
            credentials=creds,
            static_discovery=False
        )
        print(f'{API_SERVICE_NAME} {API_VERSION} service created successfully')
        return service

    except Exception as e:
        print(e)
        print(f'Failed to create service instance for {API_SERVICE_NAME}')
        os.remove(os.path.join(working_dir, token_dir, token_file))
        return None


call the service

In [ ]:
client_secret_file = 'client_secrets.json'
API_SERVICE_NAME = 'gmail'
API_VERSION = 'v1'
SCOPES = ['https://mail.google.com/']

service = create_service(client_secret_file, API_SERVICE_NAME, API_VERSION, SCOPES)


In [ ]:
dir(service)

1. fetch emails

In [ ]:
import os
import base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email import encoders
from googleapiclient import errors


def init_gmail_service(client_file, api_name='gmail', api_version='v1',
                       scopes=['https://mail.google.com/']):
    
    return create_service(client_file, api_name, api_version, scopes) # call created service function

# extract the email body from the payload
def _extract_body(payload):
    body = '<Text body not available>'

    if 'parts' in payload:
        for part in payload['parts']:
            if part.get('mimeType') == 'multipart/alternative':
                for subpart in part.get('parts', []):
                    if (
                        subpart.get('mimeType') == 'text/plain'
                        and 'data' in subpart.get('body', {})
                    ):
                        body = base64.urlsafe_b64decode(
                            subpart['body']['data']
                        ).decode('utf-8')
                        break

            elif (
                part.get('mimeType') == 'text/plain'
                and 'data' in part.get('body', {})
            ):
                body = base64.urlsafe_b64decode(
                    part['body']['data']
                ).decode('utf-8')
                break

    elif 'body' in payload and 'data' in payload['body']:
        body = base64.urlsafe_b64decode(
            payload['body']['data']
        ).decode('utf-8')

    return body

# function to get email messages with pagination and folder filtering
def get_email_messages(
    service,
    user_id='me',
    label_ids=None,
    folder_name='INBOX',
    max_results=5
):
    messages = []
    next_page_token = None

    # Resolve folder (label) name → label_id
    if folder_name:
        label_results = service.users().labels().list(userId=user_id).execute()
        labels = label_results.get('labels', [])
        folder_label_id = next(
            (label['id'] for label in labels if label['name'].lower() == folder_name.lower()),
            None
        )

        if folder_label_id:
            if label_ids:
                label_ids.append(folder_label_id)
            else:
                label_ids = [folder_label_id]
        else:
            raise ValueError(f"Folder '{folder_name}' not found.")

    # Pagination loop
    while True:
        result = (
            service.users()
            .messages()
            .list(
                userId=user_id,
                labelIds=label_ids,
                maxResults=min(500, max_results - len(messages)) if max_results else 500,
                pageToken=next_page_token
            )
            .execute()
        )

        messages.extend(result.get('messages', []))
        next_page_token = result.get('nextPageToken')

        if not next_page_token or (max_results and len(messages) >= max_results):
            break

    return messages[:max_results] if max_results else messages


def get_email_message_details(service, msg_id):
    message = service.users().messages().get(
        userId='me',
        id=msg_id,
        format='full'
    ).execute()

    payload = message.get('payload')
    headers = payload.get('headers', [])

    subject = next(
        (header['value'] for header in headers if header['name'].lower() == 'subject'),
        None
    )
    if not subject:
        subject = message.get('snippet', 'No subject')

    sender = next(
        (header['value'] for header in headers if header['name'] == 'From'),
        'No sender'
    )

    recipients = next(
        (header['value'] for header in headers if header['name'] == 'To'),
        'No recipients'
    )

    snippet = message.get('snippet', 'No snippet')

    has_attachments = any(
        (part.get('filename') for part in payload.get('parts', []) if part.get('filename'))
    )

    date = next(
        (header['value'] for header in headers if header['name'] == 'Date'),
        'No date'
    )

    star = message.get('labelIds', []).count('STARRED') > 0

    label = ', '.join(message.get('labelIds', []))

    body = _extract_body(payload)

    return {
        'subject': subject,
        'sender': sender,
        'recipients': recipients,
        'body': body,
        'snippet': snippet,
        'has_attachments': has_attachments,
        'date': date,
        'star': star,
        'label': label,
    }


how to call the above service.

In [ ]:
client_file = 'client_secrets.json'
service = init_gmail_service(client_file)

messages = get_email_messages(service, max_results=5, folder_name='INBOX')
messages

In [ ]:
# print details of the first email message
if messages:
    first_msg_id = messages[0]['id']

    details = get_email_message_details(service, first_msg_id)
    if details : 
        print ('Subject:', details['subject'])
        print ('Sender:', details['sender'])
        print ('Recipients:', details['recipients'])
        print ('Body:', details['body'])
        print ('Snippet:', details['snippet'])
        print ('Has Attachments:', details['has_attachments'])
        print ('Date:', details['date'])
        print ('Starred:', details['star'])
        print ('Label:', details['label'])


    

2. send an email with attachments

In [ ]:
def send_email(service, to, subject, body, body_type='plain', attachment_paths=None):
    
    message = MIMEMultipart()
    message['to'] = to
    message['subject'] = subject

    if body_type.lower() not in ['plain', 'html']:
        raise ValueError("body_type must be either plain or html")

    message.attach(MIMEText(body, body_type.lower()))

    if attachment_paths:
        for attachment_path in attachment_paths:
            if os.path.exists(attachment_path):
                filename = os.path.basename(attachment_path)

                with open(attachment_path, "rb") as attachment:
                    part = MIMEBase("application", "octet-stream")
                    part.set_payload(attachment.read())

                encoders.encode_base64(part)

                part.add_header(
                    "Content-Disposition",
                    f"attachment; filename= {filename}",
                )

                message.attach(part)
            else:
                raise FileNotFoundError(f"File not found - {attachment_path}")

    raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode('utf-8')

    sent_message = service.users().messages().send(
        userId='me',
        body={'raw': raw_message}
    ).execute()

    return sent_message


calling the func: sending emails

In [ ]:
from pathlib import Path
to_address = 'shamathmika.shamathmika@sjsu.edu'
email_subject = 'Test Email from Gmail POC with attachments'
email_body = 'This is a test email sent using the Gmail API and Python.'

attachment_dir = Path('C:/Users/Advait Shinde/AgenticAIFrameworks/Email-Agent/attachments')

attachment_files = list(attachment_dir.glob('*'))  # get all files in the attachments directory

print ('Attachment Files:', attachment_files)

In [ ]:
response_sent_email = send_email(
    service,
    to_address,
    email_subject,
    email_body,
    body_type='plain',
    attachment_paths= attachment_files
)

print ('Sent Email Response:', response_sent_email)

3. download attachments

In [ ]:
def download_attachments_parent(service, user_id, msg_id, target_dir):
    message = service.users().messages().get(
        userId=user_id,
        id=msg_id
    ).execute()

    for part in message['payload'].get('parts', []):
        if part.get('filename'):
            att_id = part['body']['attachmentId']
            att = service.users().messages().attachments().get(
                userId=user_id,
                messageId=msg_id,
                id=att_id
            ).execute()

            data = att['data']
            file_data = base64.urlsafe_b64decode(data.encode('UTF-8'))
            file_path = os.path.join(target_dir, part['filename'])

            print('Saving attachment to:', file_path)
            with open(file_path, 'wb') as f:
                f.write(file_data)


def download_attachments_all(service, user_id, msg_id, target_dir):
    thread = service.users().threads().get(
        userId=user_id,
        id=msg_id
    ).execute()

    for message in thread['messages']:
        for part in message['payload'].get('parts', []):
            if part.get('filename'):
                att_id = part['body']['attachmentId']
                att = service.users().messages().attachments().get(
                    userId=user_id,
                    messageId=message['id'],
                    id=att_id
                ).execute()

                data = att['data']
                file_data = base64.urlsafe_b64decode(data.encode('UTF-8'))
                file_path = os.path.join(target_dir, part['filename'])

                print('Saving attachment to:', file_path)
                with open(file_path, 'wb') as f:
                    f.write(file_data)


calling the func: downloading attachments

In [ ]:
user_id = 'me'
msg_id = '19b0a8f8d5df4124'
download_dir = 'C:/Users/Advait Shinde/AgenticAIFrameworks/Email-Agent/downloads'
download_attachments_parent(service, user_id, msg_id, download_dir)

4. searching emails

- from
- to
- cc
- bcc
- subject
- after
- before
- older
- newer
- OR
- AND


In [ ]:
def search_emails(service, query, user_id='me', max_results=5):
    messages = []
    next_page_token = None

    while True:
        result = service.users().messages().list(
            userId=user_id,
            q=query,
            maxResults=min(500, max_results - len(messages)) if max_results else 500,
            pageToken=next_page_token
        ).execute()

        messages.extend(result.get('messages', []))
        next_page_token = result.get('nextPageToken')

        if not next_page_token or (max_results and len(messages) >= max_results):
            break

    return messages[:max_results] if max_results else messages


def search_email_conversations(service, query, user_id='me', max_results=5):
    conversations = []
    next_page_token = None

    while True:
        result = service.users().threads().list(
            userId=user_id,
            q=query,
            maxResults=min(500, max_results - len(conversations)) if max_results else 500,
            pageToken=next_page_token
        ).execute()

        conversations.extend(result.get('threads', []))
        next_page_token = result.get('nextPageToken')

        if not next_page_token or (max_results and len(conversations) >= max_results):
            break

    return conversations[:max_results] if max_results else conversations


calling the func: searching emails  

In [ ]:
client_file = 'client_secrets.json'
service = init_gmail_service(client_file)

query = 'from:me to:toney.zhen@sjsu.edu'
email = search_emails(service, query, max_results=5)
email

for msg in email:
    print ('Message ID:', msg['id'])
    details = get_email_message_details(service, msg['id'])
    if details :
        print ('Subject:', details['subject'])
        print ('Sender:', details['sender'])
        print ('Recipients:', details['recipients'])
        print ('Body:', details['body'])
        print ('Snippet:', details['snippet'])
        print ('Has Attachments:', details['has_attachments'])
        print ('Date:', details['date'])
        print ('Starred:', details['star'])
        print ('Label:', details['label'])
        print ('-----------------------------------')